# Exam Analysis — Scenarios to improve the pass rate

Scenarios analysed:
- **A**: Remove questions answered by nobody
- **B**: Remove questions failed by everyone who attempted them
- **C**: Remove questions with no correct answers
- **D**: Remove questions with < 25% correct rate
- **E**: Recalculate without error penalty
- **F**: Remove questions with discrimination ≤ 0
- **Combined**: D+E, F+E

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#1a1a2e', 'axes.facecolor': '#16213e',
    'axes.edgecolor': '#e94560', 'axes.labelcolor': '#e0e0e0',
    'text.color': '#e0e0e0', 'xtick.color': '#e0e0e0',
    'ytick.color': '#e0e0e0', 'grid.color': '#2a2a4a', 'grid.alpha': 0.5,
    'font.family': 'sans-serif', 'font.size': 11,
})

COLORS = {
    'original': '#0f3460', 'scenario_a': '#e94560', 'scenario_b': '#53d8fb',
    'scenario_c': '#ffc947', 'scenario_d': '#fd79a8', 'scenario_e': '#00e676',
    'scenario_f': '#a29bfe', 'pass_line': '#00e676',
}

POINTS_CORRECT = 0.11
POINTS_WRONG = -0.05
PASS_THRESHOLD = 5.0

CSV_PATH = Path('.') / '134_36018173_ZE3IFC005200_MP5072_A-Examen 1a evaluación-cualificacións.csv'

def parse_answer(val):
    if isinstance(val, (int, float)):
        return 'correct' if val > 0 else ('wrong' if val < 0 else 'unanswered')
    s = str(val).strip().strip("'").strip('\u2018')
    if s in ('-', '', 'nan'): return 'unanswered'
    try:
        num = float(s.replace(',', '.'))
        return 'correct' if num > 0 else ('wrong' if num < 0 else 'unanswered')
    except ValueError: return 'unanswered'

def parse_score(val):
    if isinstance(val, (int, float)): return float(val)
    s = str(val).strip().strip("'").strip('\u2018')
    if s in ('-', '', 'nan'): return 0.0
    try: return float(s.replace(',', '.'))
    except ValueError: return 0.0

df = pd.read_csv(CSV_PATH)
df = df[df['Apelidos'].str.strip() != 'Media xeral'].dropna(subset=['Apelidos']).copy()
q_cols = [c for c in df.columns if c.startswith('P.')]
NUM_QUESTIONS = len(q_cols)
df['Alumno'] = df['Nome'].astype(str).str.strip() + ' ' + df['Apelidos'].astype(str).str.strip()
grade_col = [c for c in df.columns if 'ualificaci' in c][0]
df['Nota_Original'] = df[grade_col].apply(parse_score)

answer_types = pd.DataFrame(index=df.index)
scores = pd.DataFrame(index=df.index)
for col in q_cols:
    answer_types[col] = df[col].apply(parse_answer)
    scores[col] = df[col].apply(parse_score)

n_students = len(df)
grades_original = df['Nota_Original'].values

print(f'Datos cargados: {n_students} alumnos, {NUM_QUESTIONS} preguntas')
print(f'Aprobados originales: {np.sum(grades_original >= PASS_THRESHOLD)}/{n_students}')

## Grade recalculation functions

In [ ]:
def recalculate_grades(scores_df, q_cols, remove_cols=None):
    """Recalcula notas eliminando columnas (preguntas) del total."""
    keep = [c for c in q_cols if c not in (remove_cols or [])]
    n_q = len(keep)
    if n_q == 0:
        return pd.Series(0.0, index=scores_df.index)
    total = scores_df[keep].sum(axis=1)
    max_possible = POINTS_CORRECT * n_q
    return ((total / max_possible) * 10.0).clip(lower=0.0)

def recalculate_no_penalty(answer_types_df, q_cols, remove_cols=None):
    """Recalcula notas sin penalización (error = 0)."""
    keep = [c for c in q_cols if c not in (remove_cols or [])]
    n_q = len(keep)
    if n_q == 0:
        return pd.Series(0.0, index=answer_types_df.index)
    total = pd.Series(0.0, index=answer_types_df.index)
    for col in keep:
        total += np.where(answer_types_df[col] == 'correct', POINTS_CORRECT, 0.0)
    max_possible = POINTS_CORRECT * n_q
    return ((total / max_possible) * 10.0).clip(lower=0.0)

def calc_discrimination(answer_types_df, q_cols, grades):
    """Calcula el índice de discriminación para cada pregunta."""
    disc = {}
    for col in q_cols:
        correct_mask = (answer_types_df[col] == 'correct').astype(int).values
        if correct_mask.std() == 0:
            disc[col] = 0.0
        else:
            disc[col] = np.corrcoef(correct_mask, grades)[0, 1]
    return disc

print('✅ Funciones definidas')

## Scenarios A, B, C (conservative)

In [ ]:
# Estadísticas por pregunta
q_stats = {}
for i, col in enumerate(q_cols, 1):
    n_correct = (answer_types[col] == 'correct').sum()
    n_wrong = (answer_types[col] == 'wrong').sum()
    n_unanswered = (answer_types[col] == 'unanswered').sum()
    q_stats[col] = {
        'num': i,
        'pct_correct': n_correct / n_students * 100,
        'correct': n_correct, 'wrong': n_wrong, 'unanswered': n_unanswered,
        'n_answered': n_correct + n_wrong,
        'all_wrong': n_correct == 0 and n_wrong > 0,
        'none_answered': (n_correct + n_wrong) == 0,
    }

# Escenario A: nadie respondió
remove_a = [c for c, s in q_stats.items() if s['none_answered']]
grades_a = recalculate_grades(scores, q_cols, remove_a).values
n_pass_a = np.sum(grades_a >= PASS_THRESHOLD)

# Escenario B: todos los que respondieron fallaron
remove_b = [c for c, s in q_stats.items() if s['all_wrong']]
grades_b = recalculate_grades(scores, q_cols, remove_b).values
n_pass_b = np.sum(grades_b >= PASS_THRESHOLD)

# Escenario C: sin ningún acierto
remove_c = [c for c, s in q_stats.items() if s['correct'] == 0]
grades_c = recalculate_grades(scores, q_cols, remove_c).values
n_pass_c = np.sum(grades_c >= PASS_THRESHOLD)

print(f'Escenario A (nadie respondió): {len(remove_a)} preguntas eliminadas → Aprobados: {n_pass_a}/{n_students}')
print(f'Escenario B (todos fallaron):  {len(remove_b)} preguntas eliminadas → Aprobados: {n_pass_b}/{n_students}')
print(f'Escenario C (0 aciertos):       {len(remove_c)} preguntas eliminadas → Aprobados: {n_pass_c}/{n_students}')

## Scenarios D, E, F (more aggressive)

In [ ]:
# Escenario D: < 25% de acierto
remove_d = [c for c, s in q_stats.items() if s['pct_correct'] < 25]
grades_d = recalculate_grades(scores, q_cols, remove_d).values
n_pass_d = np.sum(grades_d >= PASS_THRESHOLD)

print(f'\n── Escenario D: Eliminar preguntas con < 25% de acierto')
print(f'   Preguntas eliminadas: {len(remove_d)} | Restantes: {NUM_QUESTIONS - len(remove_d)}')
for col in remove_d:
    s = q_stats[col]
    print(f'     ✗ P.{s["num"]:2d} ({s["pct_correct"]:.0f}% acierto, {s["correct"]}✓ {s["wrong"]}✗ {s["unanswered"]}○)')
print(f'   → Aprobados: {n_pass_d}/{n_students} ({n_pass_d/n_students*100:.0f}%)')

In [ ]:
# Escenario E: Sin penalización
grades_e = recalculate_no_penalty(answer_types, q_cols).values
n_pass_e = np.sum(grades_e >= PASS_THRESHOLD)

print(f'\n── Escenario E: Sin penalización por respuestas erróneas')
print(f'   Fórmula: correcto = +{POINTS_CORRECT}, error = 0.00 (en vez de {POINTS_WRONG})')
print(f'   → Aprobados: {n_pass_e}/{n_students} ({n_pass_e/n_students*100:.0f}%)')
print(f'   → Media: {np.mean(grades_e):.2f} (original: {np.mean(grades_original):.2f})')

In [ ]:
# Escenario F: Eliminar preguntas con discriminación ≤ 0
disc = calc_discrimination(answer_types, q_cols, grades_original)
remove_f = [col for col, d in disc.items() if d <= 0]
grades_f = recalculate_grades(scores, q_cols, remove_f).values
n_pass_f = np.sum(grades_f >= PASS_THRESHOLD)

print(f'\n── Escenario F: Eliminar preguntas con discriminación ≤ 0')
print(f'   Preguntas eliminadas: {len(remove_f)} | Restantes: {NUM_QUESTIONS - len(remove_f)}')
for col in remove_f:
    s = q_stats[col]
    print(f'     ✗ P.{s["num"]:2d} (disc={disc[col]:+.3f}, acierto={s["pct_correct"]:.0f}%)')
print(f'   → Aprobados: {n_pass_f}/{n_students} ({n_pass_f/n_students*100:.0f}%)')

In [ ]:
# Escenarios combinados
grades_de = recalculate_no_penalty(answer_types, q_cols, remove_d).values
n_pass_de = np.sum(grades_de >= PASS_THRESHOLD)

grades_fe = recalculate_no_penalty(answer_types, q_cols, remove_f).values
n_pass_fe = np.sum(grades_fe >= PASS_THRESHOLD)

print(f'\n── Escenario D+E: Eliminar <25% acierto + sin penalización')
print(f'   → Aprobados: {n_pass_de}/{n_students} ({n_pass_de/n_students*100:.0f}%)')
print(f'\n── Escenario F+E: Eliminar disc. ≤ 0 + sin penalización')
print(f'   → Aprobados: {n_pass_fe}/{n_students} ({n_pass_fe/n_students*100:.0f}%)')

## Full comparison table

In [ ]:
all_scenarios = [
    ('Original', grades_original, 0),
    ('A: Nadie respondió', grades_a, len(remove_a)),
    ('B: Todos fallaron', grades_b, len(remove_b)),
    ('C: 0 aciertos', grades_c, len(remove_c)),
    ('D: <25% acierto', grades_d, len(remove_d)),
    ('E: Sin penalización', grades_e, 0),
    ('F: Disc. ≤ 0', grades_f, len(remove_f)),
    ('D+E combinado', grades_de, len(remove_d)),
    ('F+E combinado', grades_fe, len(remove_f)),
]

stats_rows = []
for name, grades, n_removed in all_scenarios:
    n_pass = np.sum(grades >= PASS_THRESHOLD)
    stats_rows.append({
        'Escenario': name,
        'Elim.': n_removed,
        'Rest.': NUM_QUESTIONS - n_removed,
        'Media': round(np.mean(grades), 2),
        'Mediana': round(np.median(grades), 2),
        'Desv.Std': round(np.std(grades), 2),
        'Mín': round(np.min(grades), 2),
        'Máx': round(np.max(grades), 2),
        'Aprobados': f'{n_pass}/{n_students} ({n_pass/n_students*100:.0f}%)',
    })

pd.DataFrame(stats_rows)

## Comparative visualisation

In [ ]:
# Histogramas de escenarios clave
plot_scenarios = [
    ('Original', grades_original, COLORS['original']),
    (f'D: <25% acierto\n(−{len(remove_d)} preg.)', grades_d, COLORS['scenario_d']),
    ('E: Sin penalización', grades_e, COLORS['scenario_e']),
    (f'F: Disc. ≤ 0\n(−{len(remove_f)} preg.)', grades_f, COLORS['scenario_f']),
    (f'D+E combinado\n(−{len(remove_d)} + sin pen.)', grades_de, '#fd79a8'),
    (f'F+E combinado\n(−{len(remove_f)} + sin pen.)', grades_fe, '#00cec9'),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Distribución de Notas: Comparativa de Escenarios', fontweight='bold', fontsize=15)
bins = np.arange(0, 11, 1)

for ax, (title, grades, color) in zip(axes.flat, plot_scenarios):
    n_pass = np.sum(grades >= PASS_THRESHOLD)
    tasa = n_pass / n_students * 100
    ax.hist(grades, bins=bins, color=color, edgecolor='white', linewidth=1.2,
            alpha=0.85, rwidth=0.85)
    ax.axvline(x=PASS_THRESHOLD, color=COLORS['pass_line'], linestyle='--',
               linewidth=2, alpha=0.8, label='Aprobado (≥5)')
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('Nota')
    ax.set_ylabel('Nº Alumnos')
    ax.set_xlim(0, 10)
    ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.text(0.97, 0.95, f'Aprobados: {n_pass}/{n_students} ({tasa:.0f}%)',
            transform=ax.transAxes, ha='right', va='top', fontsize=9, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.6))
    mean_val = np.mean(grades)
    ax.axvline(x=mean_val, color='white', linestyle=':', linewidth=1.5, alpha=0.5)
    ax.text(mean_val, ax.get_ylim()[1] * 0.7, f'μ={mean_val:.1f}',
            ha='center', fontsize=9, color='white', alpha=0.8)

plt.tight_layout()
plt.show()

In [ ]:
# Tasa de aprobados — barras comparativas
fig, ax = plt.subplots(figsize=(14, 6))
fig.suptitle('Tasa de Aprobados — Comparativa Completa', fontweight='bold', fontsize=15)

labels = [r['Escenario'] for r in stats_rows]
rates = [np.sum(g >= PASS_THRESHOLD) / n_students * 100 for _, g, _ in all_scenarios]
colors_all = [COLORS['original'], COLORS['scenario_a'], COLORS['scenario_b'],
              COLORS['scenario_c'], COLORS['scenario_d'], COLORS['scenario_e'],
              COLORS['scenario_f'], '#fd79a8', '#00cec9']

bars = ax.bar(range(len(labels)), rates, color=colors_all, edgecolor='white',
              linewidth=1.5, width=0.6, alpha=0.85)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=8, rotation=20, ha='right')
for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f'{rate:.0f}%', ha='center', va='bottom', fontweight='bold', fontsize=11)
ax.set_ylabel('% Aprobados')
ax.set_ylim(0, 110)
ax.axhline(y=50, color='white', linestyle=':', linewidth=1, alpha=0.3, label='50%')
ax.grid(True, axis='y', alpha=0.3)
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Tabla de notas por alumno en cada escenario
comp = pd.DataFrame()
comp['Alumno'] = df['Alumno'].values
for name, grades, _ in all_scenarios:
    comp[name] = np.round(grades, 2)
comp

## Conclusions of the scenarios

1. **Conservative scenarios (A, B, C)** have variable impact depending on question quality. In general, removing questions that nobody answered correctly, or that everyone failed, is the most easily justifiable measure.
2. **Scenario D (remove < 25% correct)** is justifiable because a correct-answer rate below 25% is worse than random chance (with 4 options), suggesting defective or excessively difficult questions.
3. **Scenario E (no penalty)** usually has the largest individual impact. Removing the penalty recalculates grades based solely on correct answers, benefiting especially students who attempted many questions.
4. **Scenario F (discrimination ≤ 0)** removes questions that do not fulfil their evaluative purpose. The impact depends on how many questions have poor discrimination.
5. **Combinations D+E and F+E** usually offer the best results, combining removal of defective questions with removal of penalty.
6. The choice of scenario depends on the **balance between pedagogical justifiability and the desired outcome**. Scenario E is the simplest to implement (requires no question removal), while D+E offers the most complete justification.